# Optimización de Hiperparámetros v3 — Máxima Accuracy Global

**Cambios respecto a v2:**

| # | Cambio | v2 | v3 |
|:-:|:-------|:---|:---|
| 1 | Métrica de optimización | `scoring='f1_macro'` | `scoring='accuracy'` |
| 2 | SMOTE | `SMOTE(sampling_strategy={2: 300})` | **Eliminado** |
| 3 | class_weight | `class_weight='balanced'` en RF, SVM, Bagging | **Eliminado** |
| 4 | Pipeline | `ImbPipeline` (imblearn) | `Pipeline` estándar (sklearn) |
| 5 | SelectKBest | k ∈ [10, 15, 20, 25] | k ∈ [10, 15, 20, 25, 35, 45] |

> **Motivación**: El objetivo del enunciado exige maximizar la **Accuracy global**.
> Las estrategias de balanceo (SMOTE + class_weight) penalizaban la accuracy al sacrificar
> predicciones correctas en las clases mayoritarias (0 y 1, que suman el 99.4%) para intentar
> detectar la Clase 2 (0.6%). Se mantiene `SelectKBest` como filtro de irrelevancia,
> ya que en v2 demostró reducir el overfitting al descartar variables ruidosas.

## 1. Importaciones y configuración global

In [13]:
import time
import numpy as np
import pandas as pd

# Pipeline estándar de sklearn (ya no necesitamos imblearn)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif

# Modelos
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Métricas
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Configuración global
SEED: int = 42
CV_FOLDS: int = 5
N_JOBS: int = -1  # Usa todos los núcleos disponibles
SCORING: str = 'accuracy'  # v3: objetivo = máxima accuracy global

np.random.seed(SEED)

# Validación cruzada estratificada (mantiene proporciones de clase en cada fold)
cv_strategy = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

print('Imports OK')
print(f'Scoring: {SCORING}')

Imports OK
Scoring: accuracy


## 2. Carga de datos

In [14]:
# Carga de datos
train_df = pd.read_csv('data/training.csv', index_col=0)
test_df  = pd.read_csv('data/test.csv', index_col=0)

# Variables a eliminar (gemelas con |correlación| > 0.99 — filtro de redundancia del EDA)
cols_to_drop = ['bpsmt', 'csuhz', 'gwrec', 'glhls', 'bqwyz']
TARGET = 'class'

# Separar features y target
X_train = train_df.drop(columns=[TARGET] + cols_to_drop)
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET] + cols_to_drop)
y_test = test_df[TARGET]

print(f'Features tras selección: {X_train.shape[1]} (eliminadas {len(cols_to_drop)})')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'\nDistribución de clases (train):')
print(y_train.value_counts().sort_index())

Features tras selección: 45 (eliminadas 5)
X_train: (3000, 45), X_test: (2000, 45)

Distribución de clases (train):
class
0    2060
1     921
2      19
Name: count, dtype: int64


## 3. Definición de Pipelines y Rejillas de Hiperparámetros

**Estrategia v3:** Pipelines limpios sin balanceo artificial.
1. `StandardScaler` — estandarización
2. `SelectKBest(f_classif)` — filtro de irrelevancia (k en rejilla)
3. Clasificador — sin `class_weight`, sin SMOTE

Se amplía el rango de `k` a [10, 15, 20, 25, 35, 45] para explorar
si usar todas las variables (k=45) es mejor que filtrar.

### 3.1 Random Forest

In [15]:
# Pipeline: Scaler → SelectKBest → RandomForest (sin class_weight)
pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', RandomForestClassifier(random_state=SEED))
])

# Rejilla: 6 × 3 × 3 × 2 = 108 combinaciones
param_grid_rf = {
    'feature_selection__k': [10, 15, 20, 25, 35, 45],
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20],
    'classifier__max_features': ['sqrt', 'log2'],
}

print(f'Pipeline RF: {list(pipeline_rf.named_steps.keys())}')
print(f'Combinaciones: {6 * 3 * 3 * 2}')

Pipeline RF: ['scaler', 'feature_selection', 'classifier']
Combinaciones: 108


### 3.2 Gradient Boosting

In [16]:
# Pipeline: Scaler → SelectKBest → GradientBoosting
pipeline_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', GradientBoostingClassifier(random_state=SEED))
])

# Rejilla: 6 × 3 × 3 × 3 = 162 combinaciones
param_grid_gb = {
    'feature_selection__k': [10, 15, 20, 25, 35, 45],
    'classifier__n_estimators': [100, 150, 200],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 4, 5],
}

print(f'Pipeline GB: {list(pipeline_gb.named_steps.keys())}')
print(f'Combinaciones: {6 * 3 * 3 * 3}')

Pipeline GB: ['scaler', 'feature_selection', 'classifier']
Combinaciones: 162


### 3.3 AdaBoost

In [17]:
# Pipeline: Scaler → SelectKBest → AdaBoost (decision stump como estimador base)
pipeline_ada = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        random_state=SEED
    ))
])

# Rejilla: 6 × 3 × 3 = 54 combinaciones
param_grid_ada = {
    'feature_selection__k': [10, 15, 20, 25, 35, 45],
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.05, 0.1, 1.0],
}

print(f'Pipeline AdaBoost: {list(pipeline_ada.named_steps.keys())}')
print(f'Combinaciones: {6 * 3 * 3}')

Pipeline AdaBoost: ['scaler', 'feature_selection', 'classifier']
Combinaciones: 54


### 3.4 SVM con Kernel RBF

In [18]:
# Pipeline: Scaler → SelectKBest → SVM RBF (sin class_weight)
pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', SVC(
        kernel='rbf',
        random_state=SEED
    ))
])

# Rejilla: 6 × 3 × 4 = 72 combinaciones
param_grid_svm = {
    'feature_selection__k': [10, 15, 20, 25, 35, 45],
    'classifier__C': [1, 10, 100],
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1],
}

print(f'Pipeline SVM: {list(pipeline_svm.named_steps.keys())}')
print(f'Combinaciones: {6 * 3 * 4}')

Pipeline SVM: ['scaler', 'feature_selection', 'classifier']
Combinaciones: 72


### 3.5 Bagging

In [19]:
# Pipeline: Scaler → SelectKBest → Bagging (estimador base sin class_weight)
pipeline_bag = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        random_state=SEED
    ))
])

# Rejilla: 6 × 3 × 3 = 54 combinaciones
param_grid_bag = {
    'feature_selection__k': [10, 15, 20, 25, 35, 45],
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_samples': [0.7, 0.8, 1.0],
}

print(f'Pipeline Bagging: {list(pipeline_bag.named_steps.keys())}')
print(f'Combinaciones: {6 * 3 * 3}')

Pipeline Bagging: ['scaler', 'feature_selection', 'classifier']
Combinaciones: 54


## 4. Ejecución de GridSearchCV

**Configuración v3:**
- `cv=5` (StratifiedKFold con shuffle)
- `scoring='accuracy'` — objetivo: máxima exactitud global
- `n_jobs=-1` (paralelización máxima)
- `refit=True` (re-entrena con los mejores params sobre todo el train)

In [20]:
# Registro de modelos: (nombre, pipeline, rejilla)
models_config = [
    {
        'name': 'Random Forest',
        'pipeline': pipeline_rf,
        'param_grid': param_grid_rf,
    },
    {
        'name': 'Gradient Boosting',
        'pipeline': pipeline_gb,
        'param_grid': param_grid_gb,
    },
    {
        'name': 'AdaBoost',
        'pipeline': pipeline_ada,
        'param_grid': param_grid_ada,
    },
    {
        'name': 'SVM (RBF)',
        'pipeline': pipeline_svm,
        'param_grid': param_grid_svm,
    },
    {
        'name': 'Bagging',
        'pipeline': pipeline_bag,
        'param_grid': param_grid_bag,
    },
]

print(f'Modelos registrados: {[m["name"] for m in models_config]}')
total_combos = sum(
    int(np.prod([len(v) for v in m['param_grid'].values()]))
    for m in models_config
)
print(f'Total combinaciones (todos los modelos): {total_combos}')
print(f'Total fits (× {CV_FOLDS} folds): {total_combos * CV_FOLDS}')

Modelos registrados: ['Random Forest', 'Gradient Boosting', 'AdaBoost', 'SVM (RBF)', 'Bagging']
Total combinaciones (todos los modelos): 450
Total fits (× 5 folds): 2250


In [21]:
# Almacén de resultados
grid_results = []

for config in models_config:
    name = config['name']
    pipeline = config['pipeline']
    param_grid = config['param_grid']

    # Número de combinaciones en la rejilla
    n_combinations = int(np.prod([len(v) for v in param_grid.values()]))

    print(f"\n{'='*70}")
    print(f'  MODELO: {name}')
    print(f'  Combinaciones: {n_combinations} × {CV_FOLDS} folds = {n_combinations * CV_FOLDS} fits')
    print(f"{'='*70}")

    # Configurar GridSearchCV
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring=SCORING,
        n_jobs=N_JOBS,
        verbose=1,
        refit=True,
    )

    # Ejecutar búsqueda
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    elapsed = time.time() - start_time

    # Desviación estándar del mejor score en los 5 folds
    best_idx = grid_search.best_index_
    std_score = grid_search.cv_results_['std_test_score'][best_idx]

    # Mostrar resultados del modelo
    print(f'\n  ✅ Mejor score ({SCORING}): {grid_search.best_score_:.4f} ± {std_score:.4f}')
    print(f'  ⏱  Tiempo total: {elapsed:.1f}s')
    print(f'  📋 Mejores parámetros:')
    for param, value in grid_search.best_params_.items():
        # Eliminar prefijos para legibilidad
        clean_param = param.replace('classifier__', '').replace('feature_selection__', '')
        print(f'     - {clean_param}: {value}')

    # Almacenar resultados
    grid_results.append({
        'Modelo': name,
        f'Mejor {SCORING}': grid_search.best_score_,
        'Std': std_score,
        'Mejores Parámetros': grid_search.best_params_,
        'Tiempo (s)': round(elapsed, 1),
        'grid_search': grid_search,
    })


  MODELO: Random Forest
  Combinaciones: 108 × 5 folds = 540 fits
Fitting 5 folds for each of 108 candidates, totalling 540 fits


c:\Users\Josep\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



  ✅ Mejor score (accuracy): 0.7367 ± 0.0147
  ⏱  Tiempo total: 100.4s
  📋 Mejores parámetros:
     - max_depth: None
     - max_features: sqrt
     - n_estimators: 300
     - k: 25

  MODELO: Gradient Boosting
  Combinaciones: 162 × 5 folds = 810 fits
Fitting 5 folds for each of 162 candidates, totalling 810 fits


c:\Users\Josep\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



  ✅ Mejor score (accuracy): 0.7477 ± 0.0279
  ⏱  Tiempo total: 727.8s
  📋 Mejores parámetros:
     - learning_rate: 0.05
     - max_depth: 3
     - n_estimators: 200
     - k: 45

  MODELO: AdaBoost
  Combinaciones: 54 × 5 folds = 270 fits
Fitting 5 folds for each of 54 candidates, totalling 270 fits

  ✅ Mejor score (accuracy): 0.7157 ± 0.0039
  ⏱  Tiempo total: 25.0s
  📋 Mejores parámetros:
     - learning_rate: 0.1
     - n_estimators: 200
     - k: 35

  MODELO: SVM (RBF)
  Combinaciones: 72 × 5 folds = 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits

  ✅ Mejor score (accuracy): 0.7420 ± 0.0051
  ⏱  Tiempo total: 15.7s
  📋 Mejores parámetros:
     - C: 10
     - gamma: 0.01
     - k: 15

  MODELO: Bagging
  Combinaciones: 54 × 5 folds = 270 fits
Fitting 5 folds for each of 54 candidates, totalling 270 fits

  ✅ Mejor score (accuracy): 0.7377 ± 0.0098
  ⏱  Tiempo total: 212.2s
  📋 Mejores parámetros:
     - max_samples: 1.0
     - n_estimators: 150
     - k: 

## 5. Resumen comparativo

In [22]:
# Tabla resumen (sin el objeto GridSearchCV)
summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'grid_search'}
    for r in grid_results
]).sort_values(f'Mejor {SCORING}', ascending=False).reset_index(drop=True)

display(summary_df[['Modelo', f'Mejor {SCORING}', 'Std', 'Tiempo (s)']])

best = summary_df.iloc[0]
print(f"\n🏆 Mejor modelo global: {best['Modelo']} "
      f"con {SCORING} = {best[f'Mejor {SCORING}']:.4f} ± {best['Std']:.4f}")

,Modelo,Mejor accuracy,Std,Tiempo (s)
0,Gradient Boosting,0.747667,0.027881,727.8
1,SVM (RBF),0.742000,0.005099,15.7
2,Bagging,0.737667,0.009752,212.2
3,Random Forest,0.736667,0.014720,100.4
4,AdaBoost,0.715667,0.003887,25.0



🏆 Mejor modelo global: Gradient Boosting con accuracy = 0.7477 ± 0.0279


## 6. Detalle de mejores parámetros por modelo

In [23]:
for result in grid_results:
    print(f"\n{'='*50}")
    print(f"  📌 {result['Modelo']}")
    print(f"  Score: {result[f'Mejor {SCORING}']:.4f} ± {result['Std']:.4f}")
    print(f"  Parámetros óptimos:")
    for param, value in result['Mejores Parámetros'].items():
        clean_param = param.replace('classifier__', '').replace('feature_selection__', '')
        print(f"     • {clean_param}: {value}")


  📌 Random Forest
  Score: 0.7367 ± 0.0147
  Parámetros óptimos:
     • max_depth: None
     • max_features: sqrt
     • n_estimators: 300
     • k: 25

  📌 Gradient Boosting
  Score: 0.7477 ± 0.0279
  Parámetros óptimos:
     • learning_rate: 0.05
     • max_depth: 3
     • n_estimators: 200
     • k: 45

  📌 AdaBoost
  Score: 0.7157 ± 0.0039
  Parámetros óptimos:
     • learning_rate: 0.1
     • n_estimators: 200
     • k: 35

  📌 SVM (RBF)
  Score: 0.7420 ± 0.0051
  Parámetros óptimos:
     • C: 10
     • gamma: 0.01
     • k: 15

  📌 Bagging
  Score: 0.7377 ± 0.0098
  Parámetros óptimos:
     • max_samples: 1.0
     • n_estimators: 150
     • k: 45


## 7. Control de Overfitting: Brecha Train vs CV

Verificamos la diferencia entre la accuracy sobre el propio training (memorizado)
y la accuracy real del CV (generalización).

In [24]:
print(f"{'Modelo':<22} {'Train Acc':>10} {'CV Acc':>10} {'Brecha':>10}")
print('=' * 55)

for result in grid_results:
    name = result['Modelo']
    gs = result['grid_search']
    best_pipe = gs.best_estimator_

    # Accuracy sobre datos de entrenamiento (memorización)
    train_preds = best_pipe.predict(X_train)
    train_acc = accuracy_score(y_train, train_preds)
    cv_acc = result[f'Mejor {SCORING}']
    gap = train_acc - cv_acc

    print(f'{name:<22} {train_acc:>10.4f} {cv_acc:>10.4f} {gap:>+10.4f}')

Modelo                  Train Acc     CV Acc     Brecha
Random Forest              1.0000     0.7367    +0.2633
Gradient Boosting          0.8843     0.7477    +0.1367
AdaBoost                   0.7330     0.7157    +0.0173
SVM (RBF)                  0.7783     0.7420    +0.0363
Bagging                    1.0000     0.7377    +0.2623
